[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milvus-io/bootcamp/blob/master/bootcamp/RAG/search_with_jev/decide_search_stopping.ipynb)
[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/milvus-io/bootcamp/blob/master/bootcamp/RAG/search_with_jev/decide_search_stopping.ipynb)

# Decide when to stop searching with Jev

Run at most two retrieval rounds and stop as soon as the evidence supports the answer. The follow-up query is supplied by the example; Jev does not generate new search text.

## Preparation

Run locally from this directory with `uv sync --python 3.12` and `uv run jupyter lab`, or install the notebook dependencies in Colab:

In [1]:
# In Colab, uncomment this setup cell. Local users should use uv sync --python 3.12.
# %pip install "pymilvus>=2.5,<2.6.10" "milvus-lite>=2.5,<3" "setuptools<71" scikit-learn requests

> In Colab, restart the runtime after installing dependencies if needed.

Set `TYPESAFE_API_KEY` in your environment or enter it privately below. Only the small synthetic examples are sent to TypeSafe. Requests consume API credits.

The helper batches independent questions into one request. IDs map responses back to code; the instructions explicitly identify each field being judged. HTTP failures stop the tutorial rather than produce fabricated scores.

In [2]:
import getpass
import json
import math
import os
import time
import uuid

import requests
from pymilvus import DataType, MilvusClient
from sklearn.feature_extraction.text import TfidfVectorizer

if not os.getenv("TYPESAFE_API_KEY"):
    os.environ["TYPESAFE_API_KEY"] = getpass.getpass("TypeSafe API key: ")

MODEL = os.getenv("JEV_MODEL", "jev-1.13.0")
API_URL = "https://api.typesafe.ai/v1/systemone"
call_log = []

### Call Jev

This helper sends independent questions in one request and validates the returned answers.

In [3]:
def judge(state, questions):
    """Call Jev with bounded retries; stop on invalid or incomplete responses."""
    for attempt in range(3):
        started = time.perf_counter()
        response = requests.post(
            API_URL,
            headers={"Authorization": f"Bearer {os.environ['TYPESAFE_API_KEY']}"},
            json={"model": MODEL, "state": state, "questions": questions},
            timeout=45,
        )
        if response.status_code in (429, 500, 502, 503, 504) and attempt < 2:
            time.sleep(2**attempt)
            continue
        response.raise_for_status()
        body = response.json()
        answers = body["answers"]
        if set(answers) != set(questions):
            raise ValueError("Jev returned missing or unexpected question IDs")
        for key, question in questions.items():
            answer = answers[key]
            if question["type"] == "noul":
                value = float(answer["noul"])
                if not math.isfinite(value) or not 0 <= value <= 1:
                    raise ValueError("Invalid Noul probability")
            elif answer["choice"] not in question["criteria"]:
                raise ValueError("Unexpected Choice option")
        call_log.append(
            {
                "seconds": round(time.perf_counter() - started, 3),
                "usage": body.get("usage", {}),
                "model": MODEL,
            }
        )
        return answers
    raise RuntimeError("Jev request failed")


def noul(instructions):
    return {
        "type": "noul",
        "instructions": instructions,
        "criteria": {
            "true": "The stated condition is supported by the supplied data.",
            "false": "The condition is unsupported or contradicted.",
        },
    }

## Prepare a small corpus

All names and records below are synthetic teaching examples.

In [4]:
documents = [
    {
        "id": 1,
        "text": "Book A was written by author B.",
        "relation": "Book A -> written by -> B",
    },
    {"id": 2, "text": "Author B was born in city D.", "relation": "B -> born in -> D"},
    {
        "id": 3,
        "text": "Book A was published by publisher C.",
        "relation": "Book A -> published by -> C",
    },
    {"id": 4, "text": "Author B won award E.", "relation": "B -> won -> E"},
]

## Connect to Milvus

For `MilvusClient`:

- Use a local file such as `./search_with_jev.db` for [Milvus Lite](https://milvus.io/docs/milvus_lite.md).
- Set `MILVUS_URI` to a server endpoint such as `http://localhost:19530` for [Milvus on Docker or Kubernetes](https://milvus.io/docs/quickstart.md).
- For [Zilliz Cloud](https://zilliz.com/cloud), set `MILVUS_URI` to the public endpoint and `MILVUS_TOKEN` to your API key.

Each run uses its own collection name. Cleanup removes only that collection.

In [5]:
client = MilvusClient(
    uri=os.getenv("MILVUS_URI", "./search_with_jev.db"),
    token=os.getenv("MILVUS_TOKEN", ""),
)
collection_name = "jev_demo_" + uuid.uuid4().hex[:12]

## Encode the sample documents

These small examples use TF-IDF lexical vectors to avoid downloading an embedding model or needing another API key. Use the same encoder for indexing and querying; replace it with a dense embedding model for production semantic retrieval.

In [6]:
# Tiny lexical vectors keep this tutorial CPU-only and avoid a second API key.
# Replace this encoder with your production embedding model for semantic retrieval.
encoder = TfidfVectorizer()
vectors = encoder.fit_transform([row["text"] for row in documents]).toarray()

## Create the collection

Define the primary key, vector and text fields explicitly. Additional sample metadata is stored in dynamic fields. The vector index and search both use cosine similarity.

In [7]:
schema = client.create_schema(auto_id=False, enable_dynamic_field=True)
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(
    field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=vectors.shape[1]
)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=8192)
index_params = client.prepare_index_params()
index_params.add_index(
    field_name="vector", index_type="AUTOINDEX", metric_type="COSINE"
)
if not client.has_collection(collection_name=collection_name):
    client.create_collection(
        collection_name=collection_name,
        schema=schema,
        index_params=index_params,
        # consistency_level="Strong",
    )

## Insert the documents

Write the document text, metadata and vectors to Milvus.

In [8]:
client.insert(
    collection_name=collection_name,
    data=[dict(row, vector=vector.tolist()) for row, vector in zip(documents, vectors)],
)

{'insert_count': 4, 'ids': [1, 2, 3, 4], 'cost': 0}

## Retrieve candidates

Return only text and sample metadata, keeping vectors out of the Jev request. Strong consistency makes newly inserted documents searchable immediately.

In [9]:
output_fields = sorted({key for row in documents for key in row if key != "vector"})


def retrieve(query, limit=5, filter_expr=""):
    vector = encoder.transform([query]).toarray()[0]
    if not vector.any():
        return []
    hits = client.search(
        collection_name=collection_name,
        data=[vector.tolist()],
        anns_field="vector",
        limit=limit,
        filter=filter_expr,
        output_fields=output_fields,
        search_params={"metric_type": "COSINE", "params": {}},
        consistency_level="Strong",
    )[0]
    return [
        dict(
            {key: value for key, value in hit["entity"].items() if key != "vector"},
            id=hit["id"],
            retrieval_score=hit["distance"],
        )
        for hit in hits
    ]

## Search, inspect evidence, then continue or stop

The cap limits work; it does not force every query to use both rounds. Reaching the cap without evidence means abstaining.

In [10]:
question = "Where was the author of Book A born?"
planned_queries = ["Book A written author", "Author B born city"]
evidence = {}
sufficient = False
for round_number, query in enumerate(planned_queries, start=1):
    for row in retrieve(query, limit=1):
        evidence[row["id"]] = row
    answer = judge(
        {"question": question, "evidence": list(evidence.values())},
        {
            "enough": noul(
                "Does `evidence` explicitly identify both the author of Book A and that author's birthplace, so `question` can be answered without another search? Do not fill gaps from outside knowledge."
            )
        },
    )
    sufficient = answer["enough"]["noul"] >= 0.8
    print({"round": round_number, "evidence_ids": list(evidence), "stop": sufficient})
    if sufficient:
        break
print(
    "Ready to answer from evidence"
    if sufficient
    else "Budget exhausted: abstain or escalate"
)

{'round': 1, 'evidence_ids': [1], 'stop': False}


{'round': 2, 'evidence_ids': [1, 2], 'stop': True}
Ready to answer from evidence


## Inspect usage and clean up

The raw usage fields and request duration help inspect this run. They are not a latency benchmark.

In [11]:
print(json.dumps(call_log, indent=2))
client.drop_collection(collection_name=collection_name)
client.close()

[
  {
    "seconds": 0.669,
    "usage": {
      "input_tokens": 430,
      "output_tokens": 21
    },
    "model": "jev-1.13.0"
  },
  {
    "seconds": 0.597,
    "usage": {
      "input_tokens": 492,
      "output_tokens": 21
    },
    "model": "jev-1.13.0"
  }
]


## Next steps

The thresholds in this example are starting points, not calibrated production defaults. Independent questions share state but do not see each other's answers. See the [Jev primitives](https://docs.typesafe.ai/primitives) and the [cookbook index](https://github.com/milvus-io/bootcamp/blob/master/bootcamp/RAG/search_with_jev/README.md).